In [16]:
import sys
from pathlib import Path

raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

sys.path.insert(0, str(raiz / "src"))

from limpieza import aprender_reglas_de_limpieza, limpiar
from pipeline_sklearn import armar_pipeline, separar_xy
from preprocessing_pipeline import (
    cargar_tabla_completa,
    dividir_por_fecha,
    fitear_reglas_de_variables,
    transform_variables,
)

tabla = cargar_tabla_completa(raiz / "data")
train_bruto, _, _ = dividir_por_fecha(tabla)

reglas_limpieza = aprender_reglas_de_limpieza(train_bruto)
tabla_limpia = limpiar(tabla, reglas_limpieza)

train_limpio, _, _ = dividir_por_fecha(tabla_limpia)
reglas_variables = fitear_reglas_de_variables(train_limpio)
tabla_con_variables = transform_variables(tabla_limpia, reglas_variables)

entrenamiento, validacion, prueba = dividir_por_fecha(tabla_con_variables)

print("Entrenamiento:", entrenamiento.shape)
print("Validación:", validacion.shape)
print("Prueba:", prueba.shape)

Entrenamiento: (848, 342)
Validación: (182, 342)
Prueba: (182, 342)


In [17]:
from variables_interacciones import columnas_candidatas

# Las columnas se determinan a partir del tramo que se usa para ajustar.
train_modelo = entrenamiento.iloc[:-1]
columnas_elegidas = columnas_candidatas(train_modelo)

indicadores_calendario = {"es_fin_de_semana", "es_fin_de_anio"}
binarias = [c for c in columnas_elegidas if c in indicadores_calendario]
numericas = [c for c in columnas_elegidas if c not in indicadores_calendario]

X_train, y_train = separar_xy(train_modelo, columnas_elegidas)
X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
X_val, y_val = separar_xy(validacion.iloc[:-1], columnas_elegidas)

print(f"Numéricas: {len(numericas)} | Binarias: {len(binarias)}")
print(f"Entrenamiento: {X_train.shape} | Validación: {X_val.shape}")

Numéricas: 326 | Binarias: 2
Entrenamiento: (830, 328) | Validación: (176, 328)


In [18]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline

modelo = Pipeline([
    ("preprocesador", armar_pipeline(numericas, binarias)),
    ("regresor", Ridge()),
])

modelo.fit(X_train, y_train)
predicciones = modelo.predict(X_val)

print(f"Días evaluados: {len(y_val)}")
print(f"MAE de validación: {mean_absolute_error(y_val, predicciones):.2f}")

Días evaluados: 176
MAE de validación: 9283.70


In [19]:
import pandas as pd
from sklearn.metrics import mean_absolute_error

from auxiliar_functions import calcular_baselines

# Calculamos el promedio móvil con el historial de entrenamiento disponible.
historial = pd.concat([entrenamiento, validacion]).sort_values("DIA")
baselines = calcular_baselines(historial)

# Alineamos los pronósticos con las mismas filas de validación.
comparacion = baselines.loc[y_val.index].copy()
comparacion["real"] = y_val
comparacion["ridge"] = predicciones

# Conservamos solo los días con valor real y los tres pronósticos.
comparacion = comparacion.dropna(
    subset=["real", "persistencia", "media_movil", "ridge"]
)

print(f"Días evaluados en común: {len(comparacion)}")
for nombre in ["persistencia", "media_movil", "ridge"]:
    mae = mean_absolute_error(comparacion["real"], comparacion[nombre])
    print(f"MAE de {nombre}: {mae:.2f}")

Días evaluados en común: 171
MAE de persistencia: 2668.33
MAE de media_movil: 2497.65
MAE de ridge: 9401.28


# Pipeline de preparación y guardado

Este notebook comprueba cómo se preparan las variables para pronosticar el consumo de frío del día siguiente.

## 1. Preparación de los datos

Cargamos la tabla diaria y la dividimos por fecha. Aprendemos las reglas de limpieza y de creación de variables con el tramo de entrenamiento. Calculamos los rezagos y promedios móviles en orden cronológico para que los primeros días de validación puedan usar el historial disponible.

## 2. Separación de las entradas y el objetivo

Usamos dos mediciones numéricas y un indicador binario como conjunto provisional. La función `separar_xy` conserva las filas con un consumo del día siguiente conocido. Quitamos la última fila de entrenamiento y de validación porque su objetivo pertenece al tramo siguiente.

## 3. Ajuste del pipeline

Para las mediciones numéricas, el pipeline rellena faltantes con la mediana, aplica Yeo-Johnson para tratar la asimetría y estandariza las columnas para que Ridge trabaje con escalas comparables. El indicador de fin de semana pasa como cero o uno. El ajuste aprende los parámetros con entrenamiento y los reutiliza para predecir en validación.

## 4. Comparación con los pronósticos de referencia

Comparamos Ridge con la persistencia y la media móvil de siete días. La media móvil usa el historial disponible hasta el día en que se emite el pronóstico. Medimos el error absoluto medio sobre las mismas fechas de validación, donde hay un consumo real y están disponibles los tres pronósticos. Un valor menor indica un error promedio menor.